# 🗜️ Context Compression and Filtering

**Day 8 — Notebook 4 of 4 | Estimated Time: 25 minutes**

---

## What You'll Learn
- Why raw retrieved chunks contain noise that hurts generation quality
- LLM-based context compression: extracting only relevant sentences
- Sentence-level filtering with relevance scoring
- Metadata-based pre-filtering before embedding search
- Integrating compression into a complete RAG pipeline

---

## The Noise Problem

When you retrieve a chunk, you often get this:

```
Retrieved chunk (300 tokens):
┌────────────────────────────────────────────────┐
│ Background paragraph (not relevant)      ← 40% │
│ ─────────────────────────────────────────────  │
│ THE ANSWER IS HERE                       ← 20% │
│ ─────────────────────────────────────────────  │
│ Related-but-not-relevant content         ← 40% │
└────────────────────────────────────────────────┘
```

Compression extracts just the useful 20%, reducing noise and saving tokens.

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb langchain-text-splitters

In [ ]:
import sys, os, json
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from pydantic import BaseModel
from IPython.display import Markdown

from google.genai import errors

try:
    client = genai.Client(api_key=get_api_key())
except errors.APIError as e:
    print(f"Gemini API Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
else:
    MODEL_ID = "gemini-2.5-flash"
    print("✅ Ready!")
EMBEDDING_MODEL = "gemini-embedding-001"
GEN_MODEL = "gemini-2.5-flash"

---

## 1. Build a Noisy Corpus

Real-world documents have lots of context around the actual answer.  
We'll use a technical documentation corpus that mirrors this structure.

In [ ]:
# Realistic technical docs — each chunk has background + answer + tangential detail
TECH_DOCS = [
    {
        "id": "api-rate-limits",
        "category": "api",
        "title": "API Rate Limits",
        "text": """Our API platform was launched in 2019 and has undergone several major 
revisions since then, with the current architecture designed for high availability 
and horizontal scaling. The system uses a distributed token bucket algorithm.

Rate limits are enforced per API key. Free tier accounts are limited to 60 requests 
per minute (RPM) and 1,000 requests per day. Pro accounts receive 600 RPM and 
100,000 requests per day. Enterprise accounts have customisable limits negotiated 
with the sales team, typically starting at 6,000 RPM.

When you exceed a rate limit, the API returns HTTP 429 Too Many Requests. The 
response headers include Retry-After indicating how many seconds to wait. Our 
SDKs handle this automatically with exponential backoff.

The engineering team is planning to introduce burst allowances in Q3 of next year, 
which will allow short periods of higher throughput for workloads that are 
inherently bursty in nature. This feature is currently in design review.""",
    },
    {
        "id": "auth-tokens",
        "category": "api",
        "title": "Authentication Tokens",
        "text": """Authentication is a critical aspect of API security. We support several 
authentication methods to accommodate different use cases and security requirements.

API keys are the simplest method — include your key in the Authorization header 
as `Bearer YOUR_API_KEY`. API keys do not expire but can be rotated at any time 
from the dashboard. Each key is associated with your account's permissions and 
rate limits.

For user-specific access, use OAuth 2.0 with the Authorization Code flow. Access 
tokens expire after 1 hour; refresh tokens are valid for 30 days and can be used 
to obtain new access tokens without user interaction.

We recommend storing tokens in environment variables, never in source code. 
Leaked tokens can be revoked immediately from the security dashboard. Our system 
logs all token usage and can alert you to unusual patterns.""",
    },
    {
        "id": "webhooks",
        "category": "api",
        "title": "Webhooks",
        "text": """Webhooks allow your application to receive real-time notifications when 
events occur in our system, rather than polling our API repeatedly. This section 
covers registration, security verification, and retry logic.

To register a webhook, POST to /webhooks with your endpoint URL and the list of 
event types you want to receive. We support these event types: payment.completed, 
payment.failed, subscription.created, subscription.cancelled, and user.verified.

All webhook payloads are signed using HMAC-SHA256. We include the signature in 
the X-Signature-256 header. Validate this before processing to ensure the payload 
is genuine. Your secret is shown once at webhook creation and cannot be retrieved again.

Failed webhook deliveries are retried with exponential backoff: after 1 minute, 
5 minutes, 30 minutes, 2 hours, and 5 hours. If all 5 attempts fail, we send 
an alert to your registered email and disable the webhook.""",
    },
    {
        "id": "data-models",
        "category": "sdk",
        "title": "Data Models and Pagination",
        "text": """Our API follows REST conventions. Resources are returned as JSON objects 
with consistent field naming. All timestamps are ISO 8601 in UTC. Monetary values 
are integers in the smallest currency unit (e.g., pence, cents).

List endpoints return paginated results. The default page size is 20 items; the 
maximum is 100. Use the `limit` parameter to control page size and `cursor` to 
advance to the next page. Each response includes a `next_cursor` field — when this 
is null, you have reached the last page.

To fetch all records, implement cursor-based pagination in a loop: request the 
first page, process results, read next_cursor, and repeat until next_cursor is null. 
Avoid relying on page numbers as they can become inconsistent if records are 
created or deleted between requests.

Filtering is available on most list endpoints using query parameters. For example, 
`GET /payments?status=completed&created_after=2024-01-01` returns completed 
payments from 2024 onwards.""",
    },
    {
        "id": "errors",
        "category": "sdk",
        "title": "Error Handling",
        "text": """Our API uses conventional HTTP response codes. 2xx codes indicate success. 
4xx codes indicate a client error — the request was invalid or not authorised. 
5xx codes indicate a server error — please retry with backoff and contact support 
if the issue persists.

Error responses include a JSON body with three fields: `code` (a machine-readable 
string like `invalid_parameter`), `message` (human-readable description), and 
`param` (the specific parameter that caused the error, if applicable).

Common error codes: `authentication_failed` (check your API key), 
`rate_limit_exceeded` (slow down, see Retry-After header), `invalid_parameter` 
(check the request body against the schema), `resource_not_found` (check the ID),
and `insufficient_permissions` (check your account tier).

Our SDKs expose typed exceptions for each error code, making it easy to handle 
specific failure modes in your application.""",
    },
]

print(f"Corpus: {len(TECH_DOCS)} documents")
for d in TECH_DOCS:
    token_est = len(d["text"].split())
    print(f"  [{d['category']}] {d['title']} (~{token_est} words)")

In [ ]:
# Build ChromaDB index with category metadata
chroma = chromadb.Client()
col = chroma.create_collection("tech_docs", metadata={"hnsw:space": "cosine"})

texts = [d["text"] for d in TECH_DOCS]
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=texts,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)
col.add(
    ids=[d["id"] for d in TECH_DOCS],
    documents=texts,
    embeddings=[e.values for e in result.embeddings],
    metadatas=[{"title": d["title"], "category": d["category"]} for d in TECH_DOCS],
)
print(f"✅ Indexed {col.count()} documents")

---

## 2. Metadata Pre-Filtering

The cheapest form of filtering happens *before* embedding search — use metadata  
to reduce the candidate set to only relevant document categories.

In [ ]:
def retrieve_with_filter(
    query: str,
    top_k: int = 3,
    category_filter: str | None = None,
) -> list[dict]:
    """Dense retrieval with optional metadata pre-filter."""
    q_vec = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values

    where = {"category": {"$eq": category_filter}} if category_filter else None

    results = col.query(
        query_embeddings=[q_vec],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
        where=where,
    )
    return [
        {
            "id": results["ids"][0][i],
            "title": results["metadatas"][0][i]["title"],
            "category": results["metadatas"][0][i]["category"],
            "text": results["documents"][0][i],
            "score": 1 - results["distances"][0][i],
        }
        for i in range(len(results["ids"][0]))
    ]


# No filter — returns from all categories
query = "How long do OAuth refresh tokens last?"
unfiltered = retrieve_with_filter(query, top_k=3)
print(f"Unfiltered results for: '{query}'")
for r in unfiltered:
    print(f"  [{r['category']}] {r['title']} (score={r['score']:.4f})")

# With filter — only SDK docs
filtered = retrieve_with_filter(query, top_k=3, category_filter="api")
print(f"\nFiltered to 'api' category:")
for r in filtered:
    print(f"  [{r['category']}] {r['title']} (score={r['score']:.4f})")

---

## 3. LLM-Based Context Compression

**Contextual compression** extracts only the sentences from a retrieved chunk  
that are relevant to the query. Everything else is discarded.

In [ ]:
class CompressedContext(BaseModel):
    relevant_text: str    # Extracted relevant sentences — empty string if nothing relevant
    is_relevant: bool     # False if the chunk has no useful information for the query
    relevance_reason: str # Brief explanation of what was extracted and why


def compress_chunk(query: str, chunk: str) -> CompressedContext:
    """
    Extract only the sentences from `chunk` that are relevant to `query`.
    Returns the compressed text plus relevance metadata.
    """
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Extract ONLY the sentences from the document that directly help answer the query.
Do not paraphrase — copy relevant sentences verbatim.
If nothing in the document is relevant, set is_relevant to false and relevant_text to an empty string.

Query: {query}

Document:
{chunk}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=CompressedContext,
            temperature=0.0,
        ),
    )
    return CompressedContext.model_validate_json(response.text)


# Demonstrate compression on a specific chunk
query = "How long do OAuth refresh tokens last?"
auth_chunk = TECH_DOCS[1]["text"]  # Authentication Tokens doc

original_words = len(auth_chunk.split())
compressed = compress_chunk(query, auth_chunk)
compressed_words = len(compressed.relevant_text.split())

print(f"Query: '{query}'")
print(f"\nOriginal chunk ({original_words} words):")
print("-" * 50)
print(auth_chunk[:500] + "...")
print(f"\nCompressed context ({compressed_words} words, {compressed_words/original_words*100:.0f}% of original):")
print("-" * 50)
print(compressed.relevant_text)
print(f"\nReason: {compressed.relevance_reason}")

In [ ]:
# Test compression across multiple chunks for a single query
query = "What happens when I exceed the API rate limit?"
chunks = retrieve_with_filter(query, top_k=4)

print(f"Query: '{query}'\n")
print(f"Retrieved {len(chunks)} chunks. Compressing...\n")

compressed_chunks = []
for c in chunks:
    result = compress_chunk(query, c["text"])
    reduction = (1 - len(result.relevant_text.split()) / len(c["text"].split())) * 100
    print(f"  [{c['title']}]")
    print(f"    Relevant: {result.is_relevant}")
    print(f"    Reduction: {reduction:.0f}%")
    print(f"    Extracted: {result.relevant_text[:120]}..." if result.relevant_text else "    (nothing relevant)")
    if result.is_relevant:
        compressed_chunks.append({**c, "compressed_text": result.relevant_text})

print(f"\n{len(compressed_chunks)} of {len(chunks)} chunks are relevant")

---

## 4. Sentence-Level Filtering

A lighter-weight alternative to full LLM compression: score each sentence  
independently and keep only those above a threshold.

In [ ]:
def split_into_sentences(text: str) -> list[str]:
    """Simple sentence splitter — splits on period+space or newlines."""
    import re
    # Split on sentence endings, keeping reasonably long sentences
    sentences = re.split(r'(?<=[.!?])\s+|\n\n', text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]


class SentenceScore(BaseModel):
    is_relevant: bool
    score: int  # 1–3: 1=not relevant, 2=tangential, 3=directly relevant


def score_sentences(
    query: str,
    text: str,
    min_score: int = 3,
) -> list[str]:
    """Filter a chunk to only sentences scoring above min_score."""
    sentences = split_into_sentences(text)

    # Score all sentences in one batch call
    class SentenceScores(BaseModel):
        scores: list[SentenceScore]

    sentences_text = "\n".join(f"{i}. {s}" for i, s in enumerate(sentences))

    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Score each sentence for relevance to the query.
3 = Directly relevant (answers the query)
2 = Tangentially related
1 = Not relevant

Query: {query}

Sentences:
{sentences_text}

Return one score object per sentence in order.""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=SentenceScores,
            temperature=0.0,
        ),
    )
    scores = SentenceScores.model_validate_json(response.text).scores

    # Keep only high-scoring sentences
    kept = []
    for sentence, score in zip(sentences, scores):
        if score.score >= min_score:
            kept.append(sentence)

    return kept


# Demo sentence filtering on a chunk with mixed content
query = "What are the retry delays when a webhook fails?"
webhook_doc = TECH_DOCS[2]["text"]  # Webhooks doc

relevant_sentences = score_sentences(query, webhook_doc, min_score=3)

print(f"Query: '{query}'")
print(f"\nOriginal: {len(split_into_sentences(webhook_doc))} sentences")
print(f"Kept:     {len(relevant_sentences)} relevant sentences")
print("\nRelevant sentences:")
for s in relevant_sentences:
    print(f"  ✓ {s}")

---

## 5. Token Budget Management

LLM context windows are finite. When you have many retrieved chunks,  
you need a strategy to fit everything within a token budget.

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough token estimate: ~0.75 words per token (GPT-style approximation)."""
    return int(len(text.split()) / 0.75)


def budget_context(
    chunks: list[dict],
    max_tokens: int = 2000,
    strategy: str = "truncate",  # "truncate" | "compress"
    query: str = "",
) -> str:
    """
    Build a context string that fits within max_tokens.
    
    Strategies:
    - truncate: include full chunks until budget is exhausted
    - compress: compress each chunk before adding
    """
    context_parts = []
    tokens_used = 0

    for chunk in chunks:
        if strategy == "compress" and query:
            compressed = compress_chunk(query, chunk["text"])
            text = compressed.relevant_text if compressed.is_relevant else ""
        else:
            text = chunk["text"]

        if not text:
            continue

        chunk_tokens = estimate_tokens(text)
        if tokens_used + chunk_tokens > max_tokens:
            # Try truncating this chunk to fit the remaining budget
            remaining = max_tokens - tokens_used
            words = text.split()
            truncated_words = int(remaining * 0.75)
            if truncated_words > 30:
                text = " ".join(words[:truncated_words]) + "..."
                context_parts.append(f"[{chunk['title']}]\n{text}")
            break

        context_parts.append(f"[{chunk['title']}]\n{text}")
        tokens_used += chunk_tokens

    return "\n\n---\n\n".join(context_parts)


# Compare token usage: truncate vs compress
query = "What error codes does the API return and what do they mean?"
all_chunks = retrieve_with_filter(query, top_k=5)

truncated_ctx = budget_context(all_chunks, max_tokens=600, strategy="truncate")
compressed_ctx = budget_context(all_chunks, max_tokens=600, strategy="compress", query=query)

print(f"Query: '{query}'")
print(f"\nTruncate strategy: {estimate_tokens(truncated_ctx)} tokens")
print(truncated_ctx[:400] + "...")
print(f"\nCompress strategy: {estimate_tokens(compressed_ctx)} tokens")
print(compressed_ctx[:400] + "...")

---

## 6. The Complete Compression Pipeline

Putting it all together: metadata filter → dense retrieval → compression → generation.

In [ ]:
def compressed_rag(
    question: str,
    top_k: int = 4,
    category_filter: str | None = None,
    max_context_tokens: int = 1500,
    verbose: bool = True,
) -> str:
    """
    Full compression RAG pipeline:
    1. Optional metadata pre-filter
    2. Dense retrieval
    3. LLM-based compression (extract relevant sentences)
    4. Token-budget management
    5. LLM generation
    """
    # Step 1+2: Retrieve with optional metadata filter
    chunks = retrieve_with_filter(question, top_k=top_k, category_filter=category_filter)
    if verbose:
        print(f"Step 1: Retrieved {len(chunks)} chunks")
        for c in chunks:
            print(f"         [{c['score']:.3f}] {c['title']}")

    # Step 3: Compress each chunk
    compressed_chunks = []
    for c in chunks:
        result = compress_chunk(question, c["text"])
        if result.is_relevant and result.relevant_text:
            original_tokens = estimate_tokens(c["text"])
            compressed_tokens = estimate_tokens(result.relevant_text)
            compressed_chunks.append({
                **c,
                "text": result.relevant_text,
                "original_tokens": original_tokens,
                "compressed_tokens": compressed_tokens,
            })

    if verbose:
        total_orig = sum(c.get("original_tokens", 0) for c in compressed_chunks)
        total_comp = sum(c.get("compressed_tokens", 0) for c in compressed_chunks)
        reduction = (1 - total_comp / max(total_orig, 1)) * 100
        print(f"Step 2: Compressed {len(chunks)} → {len(compressed_chunks)} relevant chunks")
        print(f"        Tokens: {total_orig} → {total_comp} ({reduction:.0f}% reduction)")

    if not compressed_chunks:
        return "I could not find relevant information to answer this question."

    # Step 4: Build token-budget context
    context = budget_context(
        compressed_chunks, max_tokens=max_context_tokens, strategy="truncate"
    )

    # Step 5: Generate
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Answer the question using ONLY the provided documentation context.
Be specific and include exact values (numbers, timeframes, codes) when present.

<context>
{context}
</context>

Question: {question}""",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip()


# Test the full pipeline
answer = compressed_rag("What happens when a webhook delivery fails? How many retries are there?")
print(f"\nAnswer:\n{answer}")

In [ ]:
# Compare: standard RAG vs compressed RAG on multiple questions
def standard_rag(question: str, top_k: int = 3) -> str:
    """Plain retrieve-then-generate with no compression."""
    chunks = retrieve_with_filter(question, top_k=top_k)
    context = "\n\n".join(f"[{c['title']}]\n{c['text']}" for c in chunks)
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"Answer using ONLY this documentation:\n<context>\n{context}\n</context>\n\nQuestion: {question}",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip()


eval_questions = [
    "What is the rate limit for Pro tier accounts?",
    "How do I validate that a webhook payload is genuine?",
    "What is cursor-based pagination and when does it end?",
]

print("STANDARD RAG vs COMPRESSED RAG\n")
print("=" * 70)

for q in eval_questions:
    print(f"\n❓ {q}")
    std = standard_rag(q)
    cmp = compressed_rag(q, verbose=False)
    print(f"  Standard:   {std[:150]}")
    print(f"  Compressed: {cmp[:150]}")
    print("-" * 70)

---

## 🏋️ Exercise 1: Aggressive Compression Tuning

The compression prompt can be tuned for different use cases.  
Try making it more or less aggressive.

In [ ]:
# Exercise 1: Compression level experiment
# The default compress_chunk() extracts relevant sentences verbatim.
# Your task: create two alternative compressors:
#
#   conservative_compress(query, chunk):
#     Keep any sentence that's even slightly related (score >= 2)
#     Goal: high recall, may include some noise
#
#   aggressive_compress(query, chunk):
#     Keep ONLY sentences that DIRECTLY answer the query
#     May summarise rather than extract verbatim
#     Goal: minimum tokens, maximum precision
#
# Compare both on this chunk and query:
test_query = "What is the maximum page size for list endpoints?"
test_chunk = TECH_DOCS[3]["text"]  # Data Models and Pagination

def conservative_compress(query: str, chunk: str) -> str:
    # TODO: Implement — keep anything tangentially related
    pass

def aggressive_compress(query: str, chunk: str) -> str:
    # TODO: Implement — keep ONLY the direct answer, may summarise
    pass

# TODO: Run both and print token counts + extracted text

---

## 🏋️ Exercise 2: Build a Full Compressed RAG Chatbot

Combine everything from Day 8 into a documentation chatbot with:
- Metadata filtering (auto-detect category from question)
- Context compression
- Multi-turn conversation history

In [ ]:
# Exercise 2: Documentation chatbot with compression
# TODO:
# 1. Write detect_category(question) → "api" | "sdk" | None
#    Use Gemini to classify whether the question is about the API or SDK docs
#
# 2. Build DocChatbot class:
#    - __init__: create ChromaDB collection, history list
#    - ask(question):
#        a. detect_category(question)
#        b. retrieve_with_filter() using detected category
#        c. compress_chunk() for each result
#        d. budget_context() to stay within tokens
#        e. generate with conversation history in system prompt
#        f. append to history
#        g. return answer
#
# 3. Test a 3-turn conversation:
#    Turn 1: "What's the rate limit for Pro accounts?"
#    Turn 2: "And what happens when I hit it?"
#    Turn 3: "How do I handle that in my SDK?"

class DocChatbot:
    def __init__(self, collection):
        self.col = collection
        self.history = []

    def ask(self, question: str) -> str:
        # TODO: Implement the full pipeline
        pass


bot = DocChatbot(col)

# TODO: Run 3-turn conversation here

---

## Day 8 Recap: Advanced RAG Techniques

| Technique | What It Solves | When to Use |
|-----------|---------------|-------------|
| **Hybrid Retrieval** | Dense misses keyword matches | Technical docs, proper nouns, codes |
| **Query Transformation** | Poorly-phrased queries | Conversational UIs, complex questions |
| **Reranking** | Top-K retrieval imprecision | High-stakes answers, small corpora |
| **Metadata Filtering** | Irrelevant document categories | Multi-domain corpora |
| **Context Compression** | Noisy retrieved chunks | Long documents, token budget limits |

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [LangChain Contextual Compression](https://python.langchain.com/docs/modules/data_connection/retrievers/contextual_compression/) | Docs | LangChain's compression retriever |
| [Advanced RAG Survey](https://arxiv.org/abs/2312.10997) | Paper | Comprehensive survey of all RAG improvements |
| [Lost in the Middle](https://arxiv.org/abs/2307.03172) | Paper | Why context position matters for LLMs |

---

**Day 8 Complete! 🎉**  
You've learned hybrid retrieval, query transformation, reranking, and context compression.  
**Next:** Day 9 — Evaluation Frameworks for LLM and RAG Systems